In [1]:
import pandas as pd
import numpy as np
import keras
from keras import layers, ops
from sklearn.model_selection import KFold

import ast

In [2]:
def prepare_melodies(melodies, max_len=128, pitch_to_idx=None):
    """
    Convert list of pitch sequences into transformer-ready format.

    For each melody [p0, p1, p2, ..., pN]:
      - Input:  [PAD, PAD, ..., p0, p1, ..., p_{N-1}]
      - Target: [PAD, PAD, ..., p1, p2, ..., pN]

    Both padded/truncated to max_len.

    Args:
        melodies: list of lists, each containing MIDI pitch integers
        max_len: maximum sequence length (window size)
        pitch_to_idx: optional pre-existing vocabulary mapping.
                      If None, vocabulary is built from the melodies.
                      If provided (e.g., from training set), it is reused.
                      This ensures test data uses the same vocabulary as training.

    Returns:
        xs: input array (batch, max_len)
        ys: target array (batch, max_len)
        mask: boolean array (batch, max_len) - True where loss should be computed
        vocab_size: number of unique pitches + 1 (for PAD)
        pitch_to_idx: mapping from MIDI pitch to model index
        idx_to_pitch: mapping from model index to MIDI pitch
    """
    # Build or reuse vocabulary
    if pitch_to_idx is None:
        all_pitches = sorted(set(p for mel in melodies for p in mel))
        pitch_to_idx = {p: i + 1 for i, p in enumerate(all_pitches)}

    idx_to_pitch = {i: p for p, i in pitch_to_idx.items()}
    vocab_size = max(pitch_to_idx.values()) + 1  # +1 because index 0 is PAD

    PAD = 0

    xs = []
    ys = []
    masks = []
    skipped = 0

    for mel in melodies:
        # Convert to indices, skipping pitches not in vocabulary
        indexed = [pitch_to_idx[p] for p in mel if p in pitch_to_idx]

        if len(indexed) < 2:
            skipped += 1
            continue

        # Input: all tokens except the last
        # Target: all tokens except the first (shifted by 1)
        inp = indexed[:-1]
        tgt = indexed[1:]

        # Pad or truncate to max_len
        if len(inp) >= max_len:
            # Truncate: take last max_len tokens
            inp = inp[-max_len:]
            tgt = tgt[-max_len:]
            m = [True] * max_len
        else:
            # Pad from the left
            pad_len = max_len - len(inp)
            m = [False] * pad_len + [True] * len(inp)
            inp = [PAD] * pad_len + inp
            tgt = [PAD] * pad_len + tgt

        xs.append(inp)
        ys.append(tgt)
        masks.append(m)

    if skipped > 0:
        print(f"  Warning: skipped {skipped} melodies (too short or unknown pitches)")

    return (np.array(xs), np.array(ys), np.array(masks),
            vocab_size, pitch_to_idx, idx_to_pitch)

In [3]:
# df = pd.read_csv("unique_melodies.csv")

# melodies = df["pitch"].tolist()
# melodies = df["pitch"].apply(ast.literal_eval).tolist()

In [4]:
# xs, ys, masks, vocab_size, pitch_to_idx, idx_to_pitch = prepare_melodies(melodies, max_len=128)

# print(f"Input shape: {xs.shape}")
# print(f"Target shape: {ys.shape}")
# print(f"Mask shape: {masks.shape}")
# print(f"Vocab size: {vocab_size}")
# print(f"\nFirst melody input (last 20): {xs[0][-20:]}")
# print(f"First melody target (last 20): {ys[0][-20:]}")
# print(f"First melody mask  (last 20): {masks[0][-20:]}")
# print(f"\nPitch to index mapping (first 10): {dict(list(pitch_to_idx.items())[:10])}")

In [5]:
# Model components
class TokenAndPositionEmbedding(layers.Layer):
    """Embeds tokens and adds learned positional encoding."""

    def __init__(self, max_len, vocab_size, embed_dim):
        super().__init__()
        self.token_emb = layers.Embedding(
            input_dim=vocab_size, output_dim=embed_dim, mask_zero=False
        )
        self.pos_emb = layers.Embedding(
            input_dim=max_len, output_dim=embed_dim
        )

    def call(self, x):
        seq_len = ops.shape(x)[-1]
        positions = ops.arange(start=0, stop=seq_len, step=1)
        token_embeddings = self.token_emb(x)
        position_embeddings = self.pos_emb(positions)
        return token_embeddings + position_embeddings


class CausalTransformerBlock(layers.Layer):
    """
    Transformer block with CAUSAL attention mask.

    The causal mask ensures position i can only attend to positions <= i,
    matching IDyOM's left-to-right prediction constraint.
    """

    def __init__(self, embed_dim, num_heads, ff_dim, dropout_rate=0.1):
        super().__init__()
        self.att = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim
        )
        self.ffn = keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(embed_dim),
        ])
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(dropout_rate)
        self.dropout2 = layers.Dropout(dropout_rate)

    def call(self, inputs, training=False):
        # Causal mask: each position can only attend to itself and earlier positions
        attn_output = self.att(
            inputs, inputs, use_causal_mask=True
        )
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)

        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

In [6]:
def build_transformer(
    vocab_size,
    max_len=64,
    embed_dim=32,
    num_heads=4,
    ff_dim=64,
    num_layers=2,
    dropout_rate=0.1,
):
    """
    Build a simple causal transformer for next-pitch prediction.

    Default configuration is deliberately small for fair comparison with IDyOM:
    - embed_dim=32, num_heads=4, ff_dim=64, num_layers=2
    """
    inputs = keras.Input(shape=(max_len,))

    # Embedding
    x = TokenAndPositionEmbedding(max_len, vocab_size, embed_dim)(inputs)

    # Transformer blocks with causal masking
    for _ in range(num_layers):
        x = CausalTransformerBlock(embed_dim, num_heads, ff_dim, dropout_rate)(x)

    # Per-position prediction head
    # Dense applied to (batch, seq_len, embed_dim) works per-position automatically
    outputs = layers.Dense(vocab_size, activation="softmax")(x)

    model = keras.Model(inputs=inputs, outputs=outputs)
    return model

In [7]:
def masked_sparse_crossentropy(y_true, y_pred):
    """
    Sparse categorical crossentropy that ignores PAD tokens (index 0).

    This ensures padding doesn't artificially lower the loss.
    """
    # Create mask: True where target is not PAD
    mask = ops.cast(y_true != 0, dtype="float32")

    # Compute per-position loss
    loss = keras.losses.sparse_categorical_crossentropy(y_true, y_pred)

    # Apply mask and average over non-padded positions only
    masked_loss = loss * mask
    return ops.sum(masked_loss) / (ops.sum(mask) + 1e-8)


In [8]:
def compute_ic(model, xs, ys, masks):
    """
    Compute Information Content (IC) in bits for each note prediction.

    Args:
        model: trained transformer model
        xs: input sequences (batch, max_len)
        ys: target sequences (batch, max_len)
        masks: boolean mask (batch, max_len) - True for real notes

    Returns:
        mean_ic: mean IC in bits across all predicted notes
        all_ics: list of IC values per note (flattened)
    """
    # Get model predictions
    probs = model.predict(xs, verbose=0)

    all_ics = []

    for i in range(len(xs)):
        for t in range(len(xs[i])):
            if masks[i][t]:
                # Probability assigned to the correct next token
                true_token = ys[i][t]
                p = probs[i][t][true_token]

                # IC in bits (p is a clip to avoid log(0))
                p = max(p, 1e-10)
                ic = -np.log2(p)
                all_ics.append(ic)

    mean_ic = np.mean(all_ics)
    return mean_ic, all_ics


def compute_ic_per_melody(model, xs, ys, masks, melody_indices):
    """
    Compute mean IC per melody (useful for per-piece comparison with IDyOM).

    Args:
        melody_indices: list of melody names/IDs corresponding to each row in xs

    Returns:
        dict mapping melody_id -> mean IC in bits
    """
    probs = model.predict(xs, verbose=0)
    melody_ics = {}

    for i in range(len(xs)):
        mel_id = melody_indices[i]
        ics = []
        for t in range(len(xs[i])):
            if masks[i][t]:
                true_token = ys[i][t]
                p = max(probs[i][t][true_token], 1e-10)
                ics.append(-np.log2(p))
        melody_ics[mel_id] = np.mean(ics)

    return melody_ics


In [9]:
def train_model(
    melodies,
    max_len=128,
    embed_dim=32,
    num_heads=4,
    ff_dim=64,
    num_layers=2,
    dropout_rate=0.1,
    batch_size=32,
    epochs=50,
    validation_split=0.1,
    patience=5,
):
    """
    Simple full-corpus training (for initial testing only).
    For proper evaluation, use run_kfold or run_cross_corpus.
    """
    xs, ys, masks, vocab_size, pitch_to_idx, idx_to_pitch = \
        prepare_melodies(melodies, max_len=max_len)

    print(f"Data prepared:")
    print(f"  Melodies: {len(melodies)}")
    print(f"  Vocab size: {vocab_size} ({vocab_size - 1} pitches + PAD)")
    print(f"  Sequence length: {max_len}")
    print(f"  Input shape: {xs.shape}")

    model = build_transformer(
        vocab_size=vocab_size, max_len=max_len, embed_dim=embed_dim,
        num_heads=num_heads, ff_dim=ff_dim, num_layers=num_layers,
        dropout_rate=dropout_rate,
    )

    print(f"\n  Total parameters: {model.count_params():,}")
    model.summary()

    model.compile(
        loss=masked_sparse_crossentropy,
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    )

    early_stop = keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=patience, restore_best_weights=True,
    )

    history = model.fit(
        xs, ys,
        sample_weight=masks.astype(np.float32),
        batch_size=batch_size, epochs=epochs,
        validation_split=validation_split, callbacks=[early_stop],
    )

    return model, history, {
        "vocab_size": vocab_size,
        "pitch_to_idx": pitch_to_idx,
        "idx_to_pitch": idx_to_pitch,
        "max_len": max_len,
    }

In [10]:
#     IDyOM command per fold:
# (idyom:idyom <test-dataset-id> '(cpitch) '(cpitch) :models :ltm :pretraining-ids '(<train-dataset-id>))

def run_kfold(
    melodies,
    melody_ids=None,
    k=10,
    max_len=128,
    embed_dim=32,
    num_heads=4,
    ff_dim=64,
    num_layers=2,
    dropout_rate=0.1,
    batch_size=32,
    epochs=50,
    patience=5,
    random_state=42,
    export_folds_path=None,
):
    """
    K-fold cross-validation for fair comparison with IDyOM.

    Both models must use identical train/test splits. This function:
    1. Splits melodies into k folds at the piece level
    2. For each fold, trains on training melodies, evaluates on test melodies
    3. Exports fold assignments so IDyOM can use the same splits

    The exported CSV has columns: melody_id, fold, split
    Use it to create matching IDyOM datasets for each fold.
    """
    if melody_ids is None:
        melody_ids = list(range(len(melodies)))

    kf = KFold(n_splits=k, shuffle=True, random_state=random_state)

    all_ics = []
    fold_results = []
    fold_assignments = []

    for fold, (train_idx, test_idx) in enumerate(kf.split(melodies)):
        print(f"\n{'='*60}")
        print(f"FOLD {fold + 1}/{k}")
        print(f"{'='*60}")

        train_melodies = [melodies[i] for i in train_idx]
        test_melodies = [melodies[i] for i in test_idx]
        train_ids = [melody_ids[i] for i in train_idx]
        test_ids = [melody_ids[i] for i in test_idx]

        print(f"Train: {len(train_melodies)} melodies")
        print(f"Test:  {len(test_melodies)} melodies")

        # Record fold assignments
        for idx in train_idx:
            fold_assignments.append({
                "melody_id": melody_ids[idx],
                "fold": fold + 1,
                "split": "train",
            })
        for idx in test_idx:
            fold_assignments.append({
                "melody_id": melody_ids[idx],
                "fold": fold + 1,
                "split": "test",
            })

        # Prepare data: vocabulary built from training set only
        xs_train, ys_train, masks_train, vocab_size, pitch_to_idx, _ = \
            prepare_melodies(train_melodies, max_len=max_len)

        # Test data uses same vocabulary
        xs_test, ys_test, masks_test, _, _, _ = \
            prepare_melodies(test_melodies, max_len=max_len, pitch_to_idx=pitch_to_idx)

        print(f"Vocab size: {vocab_size}")

        # Build and train
        model = build_transformer(
            vocab_size=vocab_size, max_len=max_len, embed_dim=embed_dim,
            num_heads=num_heads, ff_dim=ff_dim, num_layers=num_layers,
            dropout_rate=dropout_rate,
        )

        if fold == 0:
            print(f"Total parameters: {model.count_params():,}")

        model.compile(
            loss=masked_sparse_crossentropy,
            optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        )

        early_stop = keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=patience, restore_best_weights=True
        )

        model.fit(
            xs_train, ys_train,
            sample_weight=masks_train.astype(np.float32),
            batch_size=batch_size, epochs=epochs,
            validation_split=0.1,  # within training set, for early stopping only
            callbacks=[early_stop], verbose=1,
        )

        # Evaluate on test fold
        mean_ic, ics = compute_ic(model, xs_test, ys_test, masks_test)
        melody_ics = compute_ic_per_melody(model, xs_test, ys_test, masks_test, test_ids)

        print(f"\nFold {fold + 1} mean IC: {mean_ic:.3f} bits")
        all_ics.extend(ics)
        fold_results.append({
            "fold": fold + 1,
            "mean_ic": mean_ic,
            "train_size": len(train_melodies),
            "test_size": len(test_melodies),
            "melody_ics": melody_ics,
        })

    # Export fold assignments for IDyOM replication
    df_folds = pd.DataFrame(fold_assignments)
    if export_folds_path:
        df_folds.to_csv(export_folds_path, index=False)
        print(f"\nFold assignments saved to: {export_folds_path}")

    # Summary
    overall_mean_ic = np.mean(all_ics)
    overall_std_ic = np.std(all_ics)

    print(f"\n{'='*60}")
    print(f"OVERALL RESULTS ({k}-fold cross-validation)")
    print(f"{'='*60}")
    print(f"Mean IC: {overall_mean_ic:.3f} bits")
    print(f"Std IC:  {overall_std_ic:.3f} bits")
    for r in fold_results:
        print(f"  Fold {r['fold']}: {r['mean_ic']:.3f} bits ({r['test_size']} test melodies)")

    return {
        "overall_mean_ic": overall_mean_ic,
        "overall_std_ic": overall_std_ic,
        "all_ics": all_ics,
        "fold_results": fold_results,
        "fold_assignments": df_folds,
    }

## Essen

In [ ]:
essen_melodies = pd.read_csv("essen_unique_melodies.csv")["pitch"].apply(ast.literal_eval).tolist()

# model, history, data_info = train_model(
#     melodies,
#     max_len=128,
#     embed_dim=32,
#     num_heads=4,
#     ff_dim=64,
#     num_layers=2,
#     epochs=10,
# )

model, history, data_info = train_model(essen_melodies, max_len=128, epochs=10)


Data prepared:
  Melodies: 8203
  Vocab size: 53 (52 pitches + PAD)
  Sequence length: 128
  Input shape: (8203, 128)

  Total parameters: 49,781


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ token_and_position_embedding    │ (None, 128, 32)        │         5,792 │
│ (TokenAndPositionEmbedding)     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ causal_transformer_block        │ (None, 128, 32)        │        21,120 │
│ (CausalTransformerBlock)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ causal_transformer_block_1      │ (None, 128, 32)        │        21,120 │
│ (CausalTransformerBlock)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128, 53)        │         1,749 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 49,781 (194.46 KB)

 Trainable params: 49,781 (194.46 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
231/231 ━━━━━━━━━━━━━━━━━━━━ 22s 45ms/step - loss: 0.8676 - val_loss: 1.0123
Epoch 2/10
231/231 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.7123 - val_loss: 0.9482
Epoch 3/10
231/231 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.6921 - val_loss: 0.9266
Epoch 4/10
231/231 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.6828 - val_loss: 0.9104
Epoch 5/10
231/231 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.6768 - val_loss: 0.9103
Epoch 6/10
231/231 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.6728 - val_loss: 0.8980
Epoch 7/10
231/231 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.6697 - val_loss: 0.8934
Epoch 8/10
231/231 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.6664 - val_loss: 0.8912
Epoch 9/10
231/231 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.6644 - val_loss: 0.8922
Epoch 10/10
231/231 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 0.6625 - val_loss: 0.8869


In [ ]:
# Compute IC
xs, ys, masks, _, _, _ = prepare_melodies(essen_melodies, max_len=128)
mean_ic, all_ics = compute_ic(model, xs, ys, masks)
print(f"\nMean IC: {mean_ic:.3f} bits")


Mean IC: 2.465 bits


In [ ]:
# results = run_kfold(
#     essen_melodies, melody_ids=None, k=10,
#     max_len=128, epochs=50,
#     export_folds_path="essen_fold_assignments.csv",
# )


FOLD 1/10
Train: 7382 melodies
Test:  821 melodies
Vocab size: 52
Total parameters: 49,716
Epoch 1/50
208/208 ━━━━━━━━━━━━━━━━━━━━ 21s 47ms/step - loss: 0.8700 - val_loss: 1.0287
Epoch 2/50
208/208 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.7166 - val_loss: 0.9590
Epoch 3/50
208/208 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - loss: 0.6956 - val_loss: 0.9370
Epoch 4/50
208/208 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.6860 - val_loss: 0.9234
Epoch 5/50
208/208 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.6804 - val_loss: 0.9162
Epoch 6/50
208/208 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.6758 - val_loss: 0.9092
Epoch 7/50
208/208 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.6722 - val_loss: 0.9056
Epoch 8/50
208/208 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.6695 - val_loss: 0.9036
Epoch 9/50
208/208 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.6670 - val_loss: 0.9006
Epoch 10/50
208/208 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.6648 - val_loss: 0.8962
Epoch 11/50
208/208 ━━━━━━━━━━━━━━━━━━━━ 1s 7

## Meertens

In [11]:
meertens_melodies = pd.read_csv("meertens_unique_melodies.csv")["pitch"].apply(ast.literal_eval).tolist()

meertens_melody_ids = pd.read_csv("meertens_unique_melodies.csv")["melody_id"].tolist()

In [18]:
# model, history, data_info = train_model(
#     melodies,
#     max_len=128,
#     embed_dim=32,
#     num_heads=4,
#     ff_dim=64,
#     num_layers=2,
#     epochs=10,
# )

meertens_model, meertens_history, meertens_data_info = train_model(
    meertens_melodies,
    max_len=180,
    epochs=50
)

Data prepared:
  Melodies: 15993
  Vocab size: 46 (45 pitches + PAD)
  Sequence length: 180
  Input shape: (15993, 180)

  Total parameters: 50,990


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 180)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ token_and_position_embedding    │ (None, 180, 32)        │         7,232 │
│ (TokenAndPositionEmbedding)     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ causal_transformer_block        │ (None, 180, 32)        │        21,120 │
│ (CausalTransformerBlock)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ causal_transformer_block_1      │ (None, 180, 32)        │        21,120 │
│ (CausalTransformerBlock)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 180, 46)        │         1,518 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 50,990 (199.18 KB)

 Trainable params: 50,990 (199.18 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
450/450 ━━━━━━━━━━━━━━━━━━━━ 23s 25ms/step - loss: 0.7597 - val_loss: 0.6837
Epoch 2/50
450/450 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - loss: 0.6839 - val_loss: 0.6655
Epoch 3/50
450/450 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - loss: 0.6710 - val_loss: 0.6585
Epoch 4/50
450/450 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - loss: 0.6644 - val_loss: 0.6528
Epoch 5/50
450/450 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - loss: 0.6601 - val_loss: 0.6489
Epoch 6/50
450/450 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - loss: 0.6568 - val_loss: 0.6459
Epoch 7/50
450/450 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - loss: 0.6539 - val_loss: 0.6444
Epoch 8/50
450/450 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - loss: 0.6515 - val_loss: 0.6411
Epoch 9/50
450/450 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - loss: 0.6491 - val_loss: 0.6395
Epoch 10/50
450/450 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - loss: 0.6470 - val_loss: 0.6378
Epoch 11/50
450/450 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - loss: 0.6448 - val_loss: 0.6357
Epoch 12/50
450/450 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/s

In [19]:
# Compute IC
xs, ys, masks, _, _, _ = prepare_melodies(meertens_melodies, max_len=180)
mean_ic, all_ics = compute_ic(meertens_model, xs, ys, masks)
print(f"\nMean IC: {mean_ic:.3f} bits")


Mean IC: 2.239 bits


In [12]:
results = run_kfold(
    meertens_melodies, melody_ids=meertens_melody_ids, k=10,
    max_len=180, epochs=75,
    export_folds_path="meertens_fold_assignments.csv",
)


FOLD 1/10
Train: 14393 melodies
Test:  1600 melodies
Vocab size: 46
Total parameters: 50,990
Epoch 1/75
405/405 ━━━━━━━━━━━━━━━━━━━━ 24s 28ms/step - loss: 0.7757 - val_loss: 0.6906
Epoch 2/75
405/405 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - loss: 0.6896 - val_loss: 0.6720
Epoch 3/75
405/405 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - loss: 0.6755 - val_loss: 0.6603
Epoch 4/75
405/405 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - loss: 0.6677 - val_loss: 0.6556
Epoch 5/75
405/405 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - loss: 0.6628 - val_loss: 0.6513
Epoch 6/75
405/405 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - loss: 0.6591 - val_loss: 0.6481
Epoch 7/75
405/405 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - loss: 0.6563 - val_loss: 0.6462
Epoch 8/75
405/405 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - loss: 0.6537 - val_loss: 0.6458
Epoch 9/75
405/405 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - loss: 0.6512 - val_loss: 0.6415
Epoch 10/75
405/405 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - loss: 0.6490 - val_loss: 0.6388
Epoch 11/75
405/405 ━━━━━━━━━━━━━━━━━━━━ 